# ONNX From Scratch: a numpy -> ONNX phrasebook

This notebook is a **teaching notebook**, not a solving notebook. Goal: after going
through it, you should be able to look at *any* numpy expression and know which
ONNX node(s) implement it, and be able to wire nodes together into a graph,
run it, and check it against numpy — on your own, for any NeuroGolf task.

It does **not** solve any numbered ARC-AGI task. The worked examples use small
synthetic arrays so the mechanics are visible without handing you an answer.
Your existing `from_scratch.ipynb` already has real, scored task solutions —
this notebook is the reference you go back to while building the *next* one.

Structure:

1. **What ONNX actually is** — the spec vs. the file format vs. the runtime, why it exists, and
   the exact protobuf structure, built by hand (with diagrams).
2. **The phrasebook** — ~25 short demos, each: numpy line -> ONNX node(s) -> run -> `assert` match.
3. **A simple graph from scratch** — a few nodes chained, one idea.
4. **Visualizing ONNX graphs** — `draw_onnx()`, Netron, `onnx_tool`, and a generic
   model -> Mermaid-diagram renderer you can call on anything.
5. **A complex graph from scratch** — many phrasebook pieces chained into one graph, mirroring the
   *shape* of a real golf solution without being one.
6. **Wiring into the real NeuroGolf tensor format** — the `(1,10,30,30)` one-hot encoding,
   `input`/`output` naming, static-shape + banned-op rules.
7. **How the cost/scoring actually works** — memory + params -> points, worked byte-by-byte on a
   real ONNX graph so you can verify the arithmetic yourself, not just take the formula on faith.
8. **Exercises** — numpy patterns with empty graph-building cells for you to fill in yourself,
   auto-checked against numpy.


## 1. What ONNX actually is

**ONNX (Open Neural Network Exchange) is not a library, not a runtime, and not "a neural
network."** It's three things bundled under one name:

1. **An operator spec** — a versioned catalog of ~200 operation definitions (`Add`, `Conv`,
   `Gather`, `Pad`, ...), each with precise math and typing rules. "Versioned" matters: you've
   already seen it above — `Squeeze`'s axes are an *attribute* at opset 12 but an *input tensor*
   at opset 13. The `opset_import` on a model pins exactly which version of every op's contract
   you mean, so a checker/runtime doesn't have to guess.
2. **An intermediate representation (IR)** — a protobuf message schema (`ModelProto`,
   `GraphProto`, `NodeProto`, ...) for describing a computation as a DAG of those ops.
3. **A file format** — `.onnx` is literally a serialized `ModelProto`. `onnx.load(path)` just
   deserializes the protobuf; there's no hidden magic.

Crucially, **ONNX doesn't execute anything**. It's a description, not an engine. Something else
(ONNX Runtime, TensorRT, CoreML, OpenVINO, ...) reads that description and actually runs it —
which is the whole point: train once wherever's convenient, then run anywhere there's a
conforming runtime, without dragging the training framework along.

```mermaid
flowchart LR
    subgraph FW["Where a graph is normally produced"]
        PT["PyTorch model"]
        TF["TensorFlow / Keras model"]
        SK["scikit-learn model"]
    end
    PT -- "torch.onnx.export()" --> ONNX
    TF -- "tf2onnx" --> ONNX
    SK -- "skl2onnx" --> ONNX
    HAND["onnx.helper.make_model()\n(hand-built graph -- what THIS repo does)"] -- "no training at all" --> ONNX
    ONNX["ONNX file\n(a serialized ModelProto)"]
    subgraph RT["Where it actually runs"]
        ORT["ONNX Runtime"]
        TRT["TensorRT"]
        CM["Core ML"]
        OV["OpenVINO"]
    end
    ONNX --> ORT
    ONNX --> TRT
    ONNX --> CM
    ONNX --> OV
```
*(If your notebook viewer doesn't render Mermaid inline, paste the block into
[mermaid.live](https://mermaid.live) or view this file in VS Code/GitHub.)*

**The insight that matters for this whole competition:** the left side of that diagram usually
has a training framework on it. It doesn't have to. ONNX is just a graph-description language —
so instead of training a model and exporting it, you can **write the `ModelProto` directly**,
by hand, node by node, with `onnx.helper`. That's not a workaround or a hack — it's exactly what
ONNX was always a description *for*. NeuroGolf is, mechanically, "write a tiny program in a
statically-shaped, DAG-only language whose instruction set is the ~200 standardized ONNX ops" —
no training, no gradient descent, ever.

### The protobuf structure

```mermaid
flowchart TD
    Model["ModelProto\n(the whole .onnx file)"] --> Opset["opset_import\n(which version of every op's spec)"]
    Model --> Graph["GraphProto"]
    Graph --> Input["input: ValueInfoProto[]\nname + dtype + STATIC shape"]
    Graph --> Output["output: ValueInfoProto[]"]
    Graph --> Init["initializer: TensorProto[]\nbaked-in constants (weights, indices, shapes...)"]
    Graph --> Nodes["node: NodeProto[]"]
    Nodes --> N1["one NodeProto =\nop_type + input names[] + output names[] + attributes"]
```

| Piece | Type | What it is |
|---|---|---|
| `model.graph` | `GraphProto` | the whole computation |
| `graph.input` / `graph.output` | `ValueInfoProto` | named, typed, *shaped* tensors that cross the graph boundary |
| `graph.node` | `NodeProto` | one operation: `op_type`, `input` names, `output` names, attributes |
| `graph.initializer` | `TensorProto` | a constant tensor (weights, indices, shapes, ...) baked into the file |
| `model.opset_import` | — | which version of each op's spec you're targeting |

A node doesn't hold data — it holds **names**. The graph is a dataflow DAG wired together purely
by matching output names to input names. `graph.initializer` entries are just tensors whose name
is already "produced" before any node runs, so a node can consume them like any other input.

Below is the smallest possible graph: one `Add` node, two runtime inputs, one output. Every
graph you build below is the same four ingredients, just more of them.


In [ ]:
import numpy as np
import onnx, onnxruntime
from onnx import helper, numpy_helper, TensorProto

# 1. describe the tensors crossing the graph boundary (name, dtype, static shape)
A = helper.make_tensor_value_info('A', TensorProto.FLOAT, [2, 2])
B = helper.make_tensor_value_info('B', TensorProto.FLOAT, [2, 2])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 2])

# 2. one node: op_type='Add', reads names ['A','B'], writes name ['Y']
add_node = helper.make_node('Add', inputs=['A', 'B'], outputs=['Y'])

# 3. wire it into a graph: (nodes, name, inputs, outputs, initializers)
graph = helper.make_graph([add_node], 'tiny_add', [A, B], [Y], [])

# 4. wrap the graph into a model with an ir_version + opset
model = helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid('', 12)])
onnx.checker.check_model(model, full_check=True)  # validates shapes/types/attrs line up

# run it exactly like any other numpy function
sess = onnxruntime.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])
a = np.array([[1, 2], [3, 4]], np.float32)
b = np.array([[10, 20], [30, 40]], np.float32)
out = sess.run(['Y'], {'A': a, 'B': b})[0]
print(out)
assert np.array_equal(out, a + b)
print("matches numpy a + b")

That whole model is exactly this dataflow graph — two named inputs, one node, one named
output:

```mermaid
flowchart LR
    A(["A  (input)"]) --> Add["Add"]
    B(["B  (input)"]) --> Add
    Add --> Y(["Y  (output)"])
```

**Two things worth internalizing right away:**

- `ir_version=10, opset 12` is the pinning used throughout this repo's real submissions
  (see `from_scratch.py`) and is what's used for every model built below — it's a proven-good
  combination against the NeuroGolf scorer, not an arbitrary choice.
- The NeuroGolf grid format is **float32**, never bool/int, because ONNX Runtime's memory
  profiler and the competition's cost model are computed over the tensor dtypes that flow through
  the graph — everything eventually gets `Cast` back to float. Keep this in mind: ops that
  naturally produce `bool` or `int64` (comparisons, `Gather` indices, `ArgMax`, ...) are usually
  *intermediate*, not final, in a golf graph.


## 2. The numpy -> ONNX phrasebook

Every demo below follows the same four-line shape: **numpy expression -> equivalent ONNX
node(s) -> run through onnxruntime -> assert it matches numpy exactly.**

Three tiny helpers remove the model/session boilerplate so each demo can focus on the one or
two nodes that actually matter. Nothing here hides node construction — you'll see every
`helper.make_node` call.


In [ ]:
import numpy as np
import onnx, onnxruntime
from onnx import helper, numpy_helper, TensorProto

def graph_from(nodes, inits, input_infos, output_infos, opset=12):
    # Wire nodes+initializers+IO into a checked ModelProto.
    g = helper.make_graph(nodes, 'g', input_infos, output_infos, inits)
    m = helper.make_model(g, ir_version=10, opset_imports=[helper.make_opsetid('', opset)])
    onnx.checker.check_model(m, full_check=True)
    return m

def run(model, feed):
    # Run a model on a dict of {input_name: numpy_array}.
    sess = onnxruntime.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])
    outs = [o.name for o in model.graph.output]
    res = sess.run(outs, feed)
    return res[0] if len(res) == 1 else res

def check(name, expected, actual, exact=True):
    expected = np.asarray(expected); actual = np.asarray(actual)
    ok = expected.shape == actual.shape and (np.array_equal(expected, actual) if exact else np.allclose(expected, actual))
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        print("  expected:\n", expected); print("  got:\n", actual)
    assert ok
    return actual

print("helpers ready: graph_from(), run(), check()")

### Elementwise arithmetic
`+ - * /` between arrays of the same (broadcastable) shape map 1:1 to `Add`/`Sub`/`Mul`/`Div`. Chain them exactly like you'd chain numpy expressions.

In [ ]:
a = np.array([[1,2],[3,4]], np.float32); b = np.array([[5,6],[7,8]], np.float32)
expected = a + b - a * b

A = helper.make_tensor_value_info('A', TensorProto.FLOAT, [2,2])
B = helper.make_tensor_value_info('B', TensorProto.FLOAT, [2,2])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2,2])
nodes = [helper.make_node('Add',['A','B'],['s']),
         helper.make_node('Mul',['A','B'],['p']),
         helper.make_node('Sub',['s','p'],['Y'])]
m = graph_from(nodes, [], [A,B], [Y])
check('a + b - a*b', expected, run(m, {'A':a,'B':b}))

### `np.where(cond, a, b)` -> `Where`
`Greater`/`Less`/`Equal` produce a **bool** tensor; `Where(cond, X, Y)` picks elementwise. This is the ONNX idiom for any conditional-per-cell rule.

In [ ]:
x = np.array([-2,-1,0,1,2], np.float32)
expected = np.where(x > 0, x, -x)   # abs(x)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [5])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [5])
inits = [numpy_helper.from_array(np.array(0.0, np.float32), 'zero')]
nodes = [helper.make_node('Greater',['X','zero'],['cond']),
         helper.make_node('Neg',['X'],['negx']),
         helper.make_node('Where',['cond','X','negx'],['Y'])]
m = graph_from(nodes, inits, [X], [Y])
check('abs(x) via Greater+Neg+Where', expected, run(m, {'X':x}))

### `.astype(...)` -> `Cast`
`Cast` has one attribute, `to`, set to the target `TensorProto` dtype enum.

In [ ]:
x = np.array([1.9, -1.9, 2.0], np.float32)
expected = x.astype(np.int64)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [3])
Y = helper.make_tensor_value_info('Y', TensorProto.INT64, [3])
nodes = [helper.make_node('Cast',['X'],['Y'], to=TensorProto.INT64)]
m = graph_from(nodes, [], [X], [Y])
check('astype(int64) via Cast', expected, run(m, {'X':x}))

### Slicing `x[a:b, c:d]` -> `Slice`
Since opset 10, `Slice` takes `starts`/`ends`/`axes`/`steps` as **input tensors** (not attributes) — so they're initializers here, exactly like every `Slice` you'll see in `from_scratch.py`.

In [ ]:
x = np.arange(20, dtype=np.float32).reshape(4,5)
expected = x[0:2, 1:4]

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [4,5])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, list(expected.shape))
inits = [numpy_helper.from_array(np.array([0,1],np.int64),'starts'),
         numpy_helper.from_array(np.array([2,4],np.int64),'ends'),
         numpy_helper.from_array(np.array([0,1],np.int64),'axes')]
nodes = [helper.make_node('Slice',['X','starts','ends','axes'],['Y'])]
m = graph_from(nodes, inits, [X], [Y])
check('x[0:2, 1:4] via Slice', expected, run(m, {'X':x}))

### `x[::-1]` (flip) -> `Slice` with a negative `steps`
Add the optional 5th `steps` input. `-1` reverses; `starts=-1, ends=-(n+1)` walks from the last element back past the first.

In [ ]:
x = np.arange(6, dtype=np.float32)
expected = x[::-1]

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [6])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [6])
inits = [numpy_helper.from_array(np.array([-1],np.int64),'starts'),
         numpy_helper.from_array(np.array([-7],np.int64),'ends'),
         numpy_helper.from_array(np.array([0],np.int64),'axes'),
         numpy_helper.from_array(np.array([-1],np.int64),'steps')]
nodes = [helper.make_node('Slice',['X','starts','ends','axes','steps'],['Y'])]
m = graph_from(nodes, inits, [X], [Y])
check('flip via negative-step Slice', expected, run(m, {'X':x}))

### Fancy indexing / permutation `x[idx]` -> `Gather`
`Gather(data, indices, axis=k)` looks up `indices` along axis `k`. This is *the* cheapest way to express any 1:1 remap (recolor tables, row/column permutations, reversed order) — one small int64 initializer, no arithmetic.

In [ ]:
x = np.array([10,20,30,40,50], np.float32)
idx = np.array([4,0,0,2], np.int64)
expected = x[idx]

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [5])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [4])
inits = [numpy_helper.from_array(idx,'idx')]
nodes = [helper.make_node('Gather',['X','idx'],['Y'], axis=0)]
m = graph_from(nodes, inits, [X], [Y])
check('x[idx] via Gather', expected, run(m, {'X':x}))

### `np.take_along_axis` (per-row/col indices) -> `GatherElements`
Unlike `Gather`, the `indices` tensor here has the **same rank** as `data` — one index per output cell, gathered along one axis. Useful when the lookup index itself varies per row.

In [ ]:
x = np.arange(9, dtype=np.float32).reshape(3,3)
idx = np.array([[0,1,2],[2,1,0],[1,1,1]], np.int64)
expected = np.take_along_axis(x, idx, axis=1)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [3,3])
IDX = helper.make_tensor_value_info('IDX', TensorProto.INT64, [3,3])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [3,3])
nodes = [helper.make_node('GatherElements',['X','IDX'],['Y'], axis=1)]
m = graph_from(nodes, [], [X, IDX], [Y])
check('take_along_axis via GatherElements', expected, run(m, {'X':x, 'IDX':idx}))

### `.reshape(...)` -> `Reshape`
Target shape is an int64 **input tensor**, not an attribute (since opset 5). `-1` for an inferred dim works exactly like in numpy.

In [ ]:
x = np.arange(12, dtype=np.float32)
expected = x.reshape(3,4)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [12])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [3,4])
inits = [numpy_helper.from_array(np.array([3,4],np.int64),'shape')]
nodes = [helper.make_node('Reshape',['X','shape'],['Y'])]
m = graph_from(nodes, inits, [X],[Y])
check('reshape(3,4)', expected, run(m,{'X':x}))

### `np.expand_dims` / `np.squeeze` -> `Unsqueeze` / `Squeeze`
At **opset 12** (what this repo pins to) the axes are attributes, e.g. `axes=[0]` — this flips to a tensor input at opset 13. Don't mix the two conventions.

In [ ]:
x = np.arange(6, dtype=np.float32)
expected = np.expand_dims(x, 0)
X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [6])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1,6])
m = graph_from([helper.make_node('Unsqueeze',['X'],['Y'],axes=[0])], [], [X],[Y])
check('expand_dims via Unsqueeze', expected, run(m,{'X':x}))

x2 = np.zeros((1,3,1), np.float32)
expected2 = np.squeeze(x2)
X2 = helper.make_tensor_value_info('X2', TensorProto.FLOAT, [1,3,1])
Y2 = helper.make_tensor_value_info('Y2', TensorProto.FLOAT, [3])
m2 = graph_from([helper.make_node('Squeeze',['X2'],['Y2'],axes=[0,2])], [], [X2],[Y2])
check('squeeze axes (0,2)', expected2, run(m2,{'X2':x2}))

### `np.broadcast_to` -> `Expand`
`Expand(data, shape)` broadcasts to a target shape — the standard building block for turning a per-row/per-column value into a full grid before combining it with something else via `Mul`/`Add`.

In [ ]:
x = np.array([[1],[2],[3]], np.float32)
expected = np.broadcast_to(x, (3,4)).copy()

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [3,1])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [3,4])
inits = [numpy_helper.from_array(np.array([3,4],np.int64),'shape')]
nodes = [helper.make_node('Expand',['X','shape'],['Y'])]
m = graph_from(nodes, inits, [X],[Y])
check('broadcast_to((3,4)) via Expand', expected, run(m,{'X':x}))

### `np.tile` -> `Tile`
`Tile(data, repeats)` repeats the *whole array* along each axis — different from `np.repeat`, which repeats *each element*.

In [ ]:
x = np.array([[1,2],[3,4]], np.float32)
expected = np.tile(x, (2,3))

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2,2])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, list(expected.shape))
inits = [numpy_helper.from_array(np.array([2,3],np.int64),'reps')]
nodes = [helper.make_node('Tile',['X','reps'],['Y'])]
m = graph_from(nodes, inits, [X],[Y])
check('tile((2,3))', expected, run(m,{'X':x}))

### `np.repeat` (elementwise) -> `Reshape` -> `Expand` -> `Reshape`
ONNX has **no direct 'repeat each element' op**. The universal trick: add a size-1 axis, `Expand` it to the repeat count, then `Reshape` it away — the extra axis merges into its neighbor. This exact 3-node pattern is how every pixel-block upscale in an ARC grid gets built.

In [ ]:
x = np.array([1,2,3], np.float32)
expected = np.repeat(x, 2)   # [1,1,2,2,3,3]

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [3])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [6])
inits = [numpy_helper.from_array(np.array([3,1],np.int64),'sh1'),   # (3,) -> (3,1)
         numpy_helper.from_array(np.array([3,2],np.int64),'sh2'),   # broadcast col to width 2
         numpy_helper.from_array(np.array([6],np.int64),'sh3')]     # flatten back to (6,)
nodes = [helper.make_node('Reshape',['X','sh1'],['r1']),
         helper.make_node('Expand',['r1','sh2'],['e']),
         helper.make_node('Reshape',['e','sh3'],['Y'])]
m = graph_from(nodes, inits, [X],[Y])
check('np.repeat(x, 2)', expected, run(m,{'X':x}))

### 2D pixel-block upscale (repeat both axes) -> same 3-node pattern, twice the axes
Insert a size-1 axis after *each* spatial axis, expand both to the block size, then collapse. This is the identical shape to `build_upscale` in `from_scratch.py` — but here it's just demonstrating the mechanic on a toy 2x2 grid, not solving anything.

In [ ]:
x = np.array([[1,2],[3,4]], np.float32)
ky, kx = 2, 3
expected = np.repeat(np.repeat(x, ky, axis=0), kx, axis=1)
H, W = x.shape

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [H,W])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [H*ky, W*kx])
inits = [numpy_helper.from_array(np.array([H,1,W,1],np.int64),'sh1'),
         numpy_helper.from_array(np.array([H,ky,W,kx],np.int64),'sh2'),
         numpy_helper.from_array(np.array([H*ky,W*kx],np.int64),'sh3')]
nodes = [helper.make_node('Reshape',['X','sh1'],['r1']),
         helper.make_node('Expand',['r1','sh2'],['e']),
         helper.make_node('Reshape',['e','sh3'],['Y'])]
m = graph_from(nodes, inits, [X],[Y])
check('2D block-upscale via Reshape->Expand->Reshape', expected, run(m,{'X':x}))

### `np.concatenate` / `np.stack` -> `Concat` (+ `Unsqueeze` for stack)
`Concat` joins along an existing axis. `np.stack` creates a *new* axis first — so `Unsqueeze` each input, then `Concat` on that new axis.

In [ ]:
a = np.array([1,2], np.float32); b = np.array([3,4,5], np.float32)
expected = np.concatenate([a,b])
A = helper.make_tensor_value_info('A', TensorProto.FLOAT, [2])
B = helper.make_tensor_value_info('B', TensorProto.FLOAT, [3])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [5])
m = graph_from([helper.make_node('Concat',['A','B'],['Y'],axis=0)], [], [A,B],[Y])
check('concatenate', expected, run(m,{'A':a,'B':b}))

a2 = np.array([1,2,3], np.float32); b2 = np.array([4,5,6], np.float32)
expected2 = np.stack([a2,b2], axis=0)
A2 = helper.make_tensor_value_info('A2', TensorProto.FLOAT, [3])
B2 = helper.make_tensor_value_info('B2', TensorProto.FLOAT, [3])
Y2 = helper.make_tensor_value_info('Y2', TensorProto.FLOAT, [2,3])
nodes2 = [helper.make_node('Unsqueeze',['A2'],['Au'],axes=[0]),
          helper.make_node('Unsqueeze',['B2'],['Bu'],axes=[0]),
          helper.make_node('Concat',['Au','Bu'],['Y2'],axis=0)]
m2 = graph_from(nodes2, [], [A2,B2],[Y2])
check('stack via Unsqueeze+Concat', expected2, run(m2,{'A2':a2,'B2':b2}))

### `.sum(axis=)` / `.max(axis=)` -> `ReduceSum` / `ReduceMax`
At opset 12, the axis list is the `axes` attribute (input-tensor form arrives at opset 13). `keepdims=0` drops the reduced axis, matching plain numpy `.sum(axis=)`.

In [ ]:
x = np.arange(12, dtype=np.float32).reshape(3,4)
expected = x.sum(axis=1)
X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [3,4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [3])
m = graph_from([helper.make_node('ReduceSum',['X'],['Y'],axes=[1],keepdims=0)], [], [X],[Y])
check('sum(axis=1)', expected, run(m,{'X':x}))

expected2 = x.max(axis=0)
Y2 = helper.make_tensor_value_info('Y2', TensorProto.FLOAT, [4])
m2 = graph_from([helper.make_node('ReduceMax',['X'],['Y2'],axes=[0],keepdims=0)], [], [X],[Y2])
check('max(axis=0)', expected2, run(m2,{'X':x}))

### `np.argmax` -> `ArgMax`
Output dtype is always `int64`. `keepdims=1` keeps the reduced axis as size 1 — set `0` to drop it, exactly like `ReduceSum` above.

In [ ]:
x = np.array([3,1,4,1,5,9,2], np.float32)
expected = np.array([np.argmax(x)])

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [7])
Y = helper.make_tensor_value_info('Y', TensorProto.INT64, [1])
m = graph_from([helper.make_node('ArgMax',['X'],['Y'],axis=0,keepdims=1)], [], [X],[Y])
check('argmax', expected, run(m,{'X':x}))

### `np.cumsum` -> `CumSum`
Since opset 11, `axis` is a **scalar int64 input**, not an attribute.

In [ ]:
x = np.array([1,2,3,4], np.float32)
expected = np.cumsum(x)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [4])
inits = [numpy_helper.from_array(np.array(0,np.int64),'axis')]
m = graph_from([helper.make_node('CumSum',['X','axis'],['Y'])], inits, [X],[Y])
check('cumsum', expected, run(m,{'X':x}))

### `np.clip` -> `Clip`
Since opset 11, `min`/`max` are input tensors (scalars), not attributes.

In [ ]:
x = np.array([-5,-1,0,3,10], np.float32)
expected = np.clip(x, 0, 5)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [5])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [5])
inits = [numpy_helper.from_array(np.array(0.0,np.float32),'lo'),
         numpy_helper.from_array(np.array(5.0,np.float32),'hi')]
m = graph_from([helper.make_node('Clip',['X','lo','hi'],['Y'])], inits, [X],[Y])
check('clip(0, 5)', expected, run(m,{'X':x}))

### `np.pad` -> `Pad`
`pads` is a single int64 tensor laid out as `[begin_axis0, begin_axis1, ..., end_axis0, end_axis1, ...]` — all begins, then all ends. This is exactly how every ARC-grid canvas-padding node in this repo pads a cropped region back out to 30x30.

In [ ]:
x = np.array([[1,2],[3,4]], np.float32)
expected = np.pad(x, ((1,0),(0,2)))

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2,2])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, list(expected.shape))
inits = [numpy_helper.from_array(np.array([1,0,0,2],np.int64),'pads')]
m = graph_from([helper.make_node('Pad',['X','pads'],['Y'])], inits, [X],[Y])
check('np.pad(((1,0),(0,2)))', expected, run(m,{'X':x}))

### One-hot encoding -> `OneHot`
`OneHot(indices, depth, values, axis=k)` — `depth` is a scalar, `values=[off_value, on_value]`. This is exactly how a recolored/derived grid gets turned back into the 10-channel one-hot NeuroGolf format.

In [ ]:
labels = np.array([0,2,1,2], np.int64)
expected = np.eye(3, dtype=np.float32)[labels]

X = helper.make_tensor_value_info('X', TensorProto.INT64, [4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [4,3])
inits = [numpy_helper.from_array(np.array(3,np.int64),'depth'),
         numpy_helper.from_array(np.array([0.0,1.0],np.float32),'values')]
m = graph_from([helper.make_node('OneHot',['X','depth','values'],['Y'],axis=-1)], inits, [X],[Y])
check('one-hot via OneHot', expected, run(m,{'X':labels}))

### `a @ b` / `np.matmul` -> `MatMul` (or `Gemm` for affine `aX+b`)
`MatMul` mirrors numpy's `@` exactly, including batched/broadcast matmul semantics. `Gemm` additionally fuses a bias-add and scalars alpha/beta — reach for it when you need `alpha * A@B + beta * C` in one node instead of `MatMul` + `Add`.

In [ ]:
rng = np.random.RandomState(0)
a = rng.randn(3,4).astype(np.float32); b = rng.randn(4,2).astype(np.float32)
expected = a @ b

A = helper.make_tensor_value_info('A', TensorProto.FLOAT, [3,4])
B = helper.make_tensor_value_info('B', TensorProto.FLOAT, [4,2])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [3,2])
m = graph_from([helper.make_node('MatMul',['A','B'],['Y'])], [], [A,B],[Y])
check('a @ b', expected, run(m,{'A':a,'B':b}), exact=False)

### Splitting into fixed-size chunks -> `Split`
At opset 12, `split` (the chunk sizes) and `axis` are attributes — one output tensor per chunk. This flips to an optional input at opset 13.

In [ ]:
x = np.arange(10, dtype=np.float32)
e1, e2 = x[:4], x[4:]

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [10])
Y1 = helper.make_tensor_value_info('Y1', TensorProto.FLOAT, [4])
Y2 = helper.make_tensor_value_info('Y2', TensorProto.FLOAT, [6])
m = graph_from([helper.make_node('Split',['X'],['Y1','Y2'],axis=0,split=[4,6])], [], [X],[Y1,Y2])
o1, o2 = run(m,{'X':x})
check('split part 1', e1, o1); check('split part 2', e2, o2)

### Sliding-window sum -> `Conv` with a ones kernel
Any "look at each cell's neighborhood and combine" numpy pattern (manual loop, `stride_tricks`, etc.) collapses to a **single `Conv` node** with a fixed weight tensor. `Conv` expects `(N, C, *spatial)` layout — hence the reshape to `(1,1,6)` here, matching the ARC grid's `(1,10,30,30)` layout.

In [ ]:
x = np.array([1,2,3,4,5,6], np.float32)
w = np.array([1,1,1], np.float32)
expected = np.array([x[i:i+3].sum() for i in range(4)], np.float32)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1,1,6])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1,1,4])
inits = [numpy_helper.from_array(w.reshape(1,1,3),'W')]
m = graph_from([helper.make_node('Conv',['X','W'],['Y'],kernel_shape=[3])], inits, [X],[Y])
check('sliding-window sum via Conv', expected.reshape(1,1,4), run(m,{'X':x.reshape(1,1,6)}))

### Combining boolean conditions -> `Greater`/`Equal` + `And`/`Or`/`Not` + `Cast`
Build each condition as a bool tensor, combine with the logical ops, then `Cast` back to float for anything downstream (multiply-by-mask, `Where`, ...).

In [ ]:
a = np.array([1,0,1,0], np.float32); b = np.array([1,1,0,0], np.float32)
expected = np.logical_and(a>0, b>0).astype(np.float32)

A = helper.make_tensor_value_info('A', TensorProto.FLOAT, [4])
B = helper.make_tensor_value_info('B', TensorProto.FLOAT, [4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [4])
inits = [numpy_helper.from_array(np.array(0.0,np.float32),'zero')]
nodes = [helper.make_node('Greater',['A','zero'],['ca']),
         helper.make_node('Greater',['B','zero'],['cb']),
         helper.make_node('And',['ca','cb'],['cand']),
         helper.make_node('Cast',['cand'],['Y'],to=TensorProto.FLOAT)]
m = graph_from(nodes, inits, [A,B],[Y])
check('(a>0) & (b>0) via Greater+And+Cast', expected, run(m,{'A':a,'B':b}))

### The static-shape gotcha: what you *can't* do

NeuroGolf bans `Loop`, `Scan`, `NonZero`, `Unique`, `Compress`, `Script`, `Function`, and any
`Sequence*` op (see `_EXCLUDED_OP_TYPES` in `neurogolf_utils.py`), and every tensor shape must be
static (no `dim_param`, no `<= 0` dims). That rules out the numpy patterns you'd reach for
first:

- `x[x > 0]` (boolean-mask compaction) — output size depends on data, so it needs `NonZero`/`Compress`. **Banned.**
- `np.unique(x)` — same problem. **Banned.**
- Python `for`/`while` loops over a data-dependent count — needs `Loop`/`Scan`. **Banned.**

The universal workaround: don't *select* a variable-size subset — instead **compute a
fixed-size result and multiply/`Where` it against a mask.** Every phrasebook demo above that
uses `Greater`+`Where`, or `Mul` by a 0/1 mask, is exactly that workaround. Whenever a numpy
solution reaches for boolean-mask indexing, first ask: "can I keep it dense with a mask
`Mul` instead?"


## 3. A simple graph from scratch

One rule, three nodes, on a toy 1D array: **replace every value below a threshold with -1,
otherwise leave it unchanged.** This is just `Where` from the phrasebook, but built end-to-end
here with its own inputs/outputs from a blank slate — the shape you'll copy for your first
from-scratch attempt at a real task.


In [ ]:
x = np.array([5, -3, 0, 8, -1, 2], np.float32)
threshold = 1.0
expected = np.where(x < threshold, -1.0, x)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [6])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [6])
inits = [numpy_helper.from_array(np.array(threshold, np.float32), 'thr'),
         numpy_helper.from_array(np.array(-1.0, np.float32), 'neg_one')]
nodes = [
    helper.make_node('Less', ['X', 'thr'], ['below']),          # bool mask: x < threshold
    helper.make_node('Where', ['below', 'neg_one', 'X'], ['Y']),  # pick -1 where below, else x
]
m = graph_from(nodes, inits, [X], [Y])
check('threshold-replace', expected, run(m, {'X': x}))

In [ ]:
# visualize it — small helper reused from from_scratch.py, generic to any model
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

def draw_onnx(path_or_model, ax=None, title=None, max_nodes=45):
    mm = onnx.load(path_or_model) if isinstance(path_or_model, str) else path_or_model
    nodes = list(mm.graph.node); own = ax is None
    if own: fig, ax = plt.subplots(figsize=(8, 4))
    if len(nodes) > max_nodes:
        c = Counter(n.op_type for n in nodes).most_common()
        ax.barh([k for k, _ in c][::-1], [v for _, v in c][::-1], color="#8e44ad")
        ax.set_title(f"{title or ''} ({len(nodes)} nodes)", fontsize=9)
        if own: plt.tight_layout(); plt.show()
        return
    prod = {o: i for i, n in enumerate(nodes) for o in n.output}
    preds = {i: [prod[x] for x in n.input if x in prod] for i, n in enumerate(nodes)}
    layer = {}
    def L(i):
        if i not in layer: layer[i] = 0 if not preds[i] else max(L(p) for p in preds[i]) + 1
        return layer[i]
    for i in range(len(nodes)): L(i)
    byl = defaultdict(list)
    for i, l in layer.items(): byl[l].append(i)
    pos = {}
    for l, ids in byl.items():
        for k, i in enumerate(ids): pos[i] = (l, -(k - (len(ids) - 1) / 2))
    for i in range(len(nodes)):
        x2, y2 = pos[i]
        for p in preds[i]:
            x1, y1 = pos[p]
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#bbb", lw=0.7))
    for i, n in enumerate(nodes):
        x, y = pos[i]
        ax.text(x, y, n.op_type, ha="center", va="center", fontsize=7,
                bbox=dict(boxstyle="round,pad=0.3", fc="#dceefb", ec="#2980b9", lw=0.8))
    ax.set_title(f"{title or 'graph'} ({len(nodes)} nodes)", fontsize=9)
    ax.axis("off"); ax.set_xlim(-0.6, max(byl) + 0.6)
    if own: plt.tight_layout(); plt.show()

draw_onnx(m, title="threshold-replace")

## 4. Visualizing ONNX graphs

`draw_onnx()` above is the "good enough, no setup" option for a graph you just hand-built. Three
other tools worth knowing, in the order you'd actually reach for them:

- **`draw_onnx()` (just used above)** — inline matplotlib, zero install, good for the small/medium
  graphs you build in this notebook. Falls back to an op-type bar chart past `max_nodes` (default
  45) since a node-and-arrow layout gets unreadable beyond that.
- **Netron** — the standard tool for *real* graphs (a 200+-node golfed solution, a baseline in
  `baseline_v22/`). Interactive: zoom, click a node to see its attributes, click a tensor to see
  its shape/dtype. Either:
  ```bash
  pip install netron
  ```
  ```python
  import netron
  netron.start("repairs/task001.onnx")   # opens a browser tab; netron.stop() when done
  ```
  or skip installing anything and drag-and-drop the `.onnx` file at
  [netron.app](https://netron.app) — it runs entirely in the browser, nothing is uploaded.
- **`onnx_tool`** — not a picture, a **table**: per-node op_type, shape, MACs, memory. When what
  you actually want is "which node is expensive" rather than "what does the graph look like,"
  this is more directly useful than any diagram — and it's already used by
  `neurogolf_utils.verify_network()` in exactly this way.
  ```python
  import onnx_tool
  onnx_tool.model_profile("repairs/task001.onnx")
  ```

### A fourth option: generate a Mermaid diagram from any model, in code

Mermaid is a text-based diagram language (`flowchart TD ...`) that a lot of markdown renderers
draw natively (GitHub, VS Code, JupyterLab, this very notebook's markdown cells above). Since it's
just text, you can **generate it programmatically from a `ModelProto`** — turning "what does this
graph look like" into a one-line function call for *any* model, not just ones you hand-authored
for a diagram.


In [ ]:
import re

def onnx_to_mermaid(model, direction="LR", show_initializers=False):
    # Walk a ModelProto's node list and emit a Mermaid flowchart string.
    g = model.graph
    def sid(name): return "n_" + re.sub(r"\W", "_", name)   # mermaid node ids can't have dots/slashes
    lines = [f"flowchart {direction}"]
    producer = {}
    for inp in g.input:
        lines.append(f'    {sid(inp.name)}(["{inp.name}"])')
        producer[inp.name] = sid(inp.name)
    init_names = {init.name for init in g.initializer}
    if show_initializers:
        for name in init_names:
            lines.append(f'    {sid(name)}[("{name}")]')
            producer[name] = sid(name)
    for i, node in enumerate(g.node):
        nid = f"op_{i}"
        lines.append(f'    {nid}["{node.op_type}"]')
        for o in node.output:
            producer[o] = nid
    for i, node in enumerate(g.node):
        nid = f"op_{i}"
        for x in node.input:
            if not show_initializers and x in init_names:
                continue   # skip constants by default -- keeps the diagram to the actual dataflow
            if x in producer:
                lines.append(f"    {producer[x]} --> {nid}")
    for out in g.output:
        lines.append(f'    {sid(out.name)}(["{out.name}"])')
        if out.name in producer:
            lines.append(f"    {producer[out.name]} --> {sid(out.name)}")
    return "\n".join(lines)

print(onnx_to_mermaid(m))   # 'm' is the threshold-replace model from Section 3 -- just text

In [ ]:
import html as _html, warnings
from IPython.display import HTML, display

MERMAID_CDN = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"

def render_mermaid(diagram, height=320):
    # Render a Mermaid diagram string live, via an iframe (needs internet for the CDN script).
    doc = (
        '<!doctype html><html><body style="margin:0;background:transparent">'
        '<div class="mermaid" style="text-align:center">' + diagram + '</div>'
        '<script src="' + MERMAID_CDN + '"></script>'
        '<script>mermaid.initialize({startOnLoad:true, theme:"neutral"});</script>'
        '</body></html>'
    )
    escaped = _html.escape(doc, quote=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # IPython nudges toward IFrame, which can't take raw HTML via srcdoc
        display(HTML(f'<iframe srcdoc="{escaped}" style="width:100%;height:{height}px;border:0;"></iframe>'))

render_mermaid(onnx_to_mermaid(m))

This works on any model, at any point in the notebook — try
`render_mermaid(onnx_to_mermaid(m_gather))` once you reach Section 7's `m_gather`, or run it on
the complex 4-stage pipeline built next. Same node-count caveat as `draw_onnx()` applies: past a
few dozen nodes a flowchart layout gets cluttered — reach for Netron once a graph is that big.


## 5. A complex graph from scratch

Now chain several phrasebook pieces into one graph on a toy grid, mirroring the *shape* of a real
golf solution (several stages, mixed op families) without being an answer to any task:

**Recipe:** flip a small grid left-right, upscale each cell into a `2x2` block, then recolor
through a fixed permutation table, then pad the result onto a bigger fixed canvas.

Each stage below is checked against numpy independently before being chained, so if something's
wrong you know exactly which stage broke.


In [ ]:
grid = np.array([[1, 2, 3],
                  [4, 5, 6]], np.float32)
H, W = grid.shape
ky, kx = 2, 2
recolor = {0: 0, 1: 7, 2: 8, 3: 9, 4: 1, 5: 2, 6: 3}   # arbitrary toy remap
canvas_h, canvas_w = 10, 10

# --- reference numpy pipeline, stage by stage ---
flipped = grid[:, ::-1]
upscaled = np.repeat(np.repeat(flipped, ky, axis=0), kx, axis=1)
recolored = np.vectorize(lambda v: recolor[int(v)])(upscaled).astype(np.float32)
expected = np.pad(recolored, ((0, canvas_h - upscaled.shape[0]), (0, canvas_w - upscaled.shape[1])))
print("numpy reference result:\n", expected)

In [ ]:
# --- the same pipeline as ONE ONNX graph ---
X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [H, W])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [canvas_h, canvas_w])
inits, nodes = [], []
def K(name, arr):
    inits.append(numpy_helper.from_array(np.asarray(arr), name)); return name

# stage 1: flip left-right via Gather with a reversed column index
nodes.append(helper.make_node('Gather', ['X', K('flip_idx', np.arange(W - 1, -1, -1, dtype=np.int64))], ['flipped'], axis=1))

# stage 2: 2x block upscale via Reshape -> Expand -> Reshape (same trick as the phrasebook demo)
K('sh1', [H, 1, W, 1]); K('sh2', [H, ky, W, kx]); K('sh3', [H * ky, W * kx])
nodes += [helper.make_node('Reshape', ['flipped', 'sh1'], ['r1']),
          helper.make_node('Expand', ['r1', 'sh2'], ['e']),
          helper.make_node('Reshape', ['e', 'sh3'], ['upscaled'])]

# stage 3: recolor via a lookup table -> Cast to int -> Gather
table = np.array([recolor.get(c, c) for c in range(10)], np.float32)
K('table', table)
nodes += [helper.make_node('Cast', ['upscaled'], ['up_i'], to=TensorProto.INT64),
          helper.make_node('Gather', ['table', 'up_i'], ['recolored'], axis=0)]

# stage 4: pad onto the fixed canvas
K('pads', [0, 0, canvas_h - H * ky, canvas_w - W * kx])
nodes.append(helper.make_node('Pad', ['recolored', 'pads'], ['Y']))

m = graph_from(nodes, inits, [X], [Y])
out = check('flip -> upscale -> recolor -> pad (chained)', expected, run(m, {'X': grid}))
draw_onnx(m, title="complex toy pipeline (4 stages)")

Notice the pattern: **each stage is one phrasebook idiom**, and the only new skill is
naming outputs so stage *N*'s output name becomes stage *N+1*'s input name. A "complex" ONNX
graph is just many simple ones wired end to end — there's no separate complex-graph API.


## 6. Wiring into the real NeuroGolf tensor format

Every submitted network takes exactly one input named **`input`** and produces exactly one
output named **`output`**, both shaped `(1, 10, 30, 30)` float32 — a **one-hot grid**: channel
`c`, row `r`, col `col` is `1.0` iff that cell's color is `c`. `neurogolf_utils.py` (already in
`data/neurogolf_utils/`) has the exact conversion + scorer this repo's submissions are checked
against, so reuse it rather than reinventing it.


In [ ]:
import sys, copy, math, json
sys.path.insert(0, "data/neurogolf_utils")
import neurogolf_utils as ngu

def load_task(t):
    return json.load(open(f"data/task{t:03d}.json"))

# ngu.convert_to_numpy turns one {"input":[[...]], "output":[[...]]} example into
# {"input": (1,10,30,30) float32, "output": (1,10,30,30) float32} one-hot tensors.
example = load_task(1)["train"][0]
benchmark = ngu.convert_to_numpy(example)
print("input tensor shape:", benchmark["input"].shape, " dtype:", benchmark["input"].dtype)
print("a cell's one-hot column sums to 1:", benchmark["input"][0, :, 0, 0].sum())

Two helpers you'll want for testing *your own* task graphs once you start finding
algorithms yourself:

- `verify_and_score(model, t)` — runs your model on every train/test/arc-gen example of task
  `t`, returns `(n_fail, cost, points)`. `n_fail == 0` means it's correct; `cost` is
  `memory_bytes + params`, and `points = max(1, 25 - ln(cost))`.
- `draw_onnx(model_or_path)` from Section 3 — to compare your graph's shape against a baseline
  in `baseline_v22/` the same way `from_scratch.ipynb` does.

This function is intentionally the same shape as the one in `from_scratch.py` — copy it over
there once you're ready to actually attempt a task; it's reproduced here so this notebook stays
runnable on its own.


In [ ]:
def verify_and_score(model, t):
    san = ngu.sanitize_model(copy.deepcopy(model))
    onnx.checker.check_model(san, full_check=True)
    o = onnxruntime.SessionOptions(); o.enable_profiling = True; o.log_severity_level = 3
    o.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
    o.profile_file_prefix = "tutorial_vs"
    s = onnxruntime.InferenceSession(san.SerializeToString(), o)
    d = load_task(t); nf = 0
    for ex in d["train"] + d["test"] + d["arc-gen"]:
        b = ngu.convert_to_numpy(ex)
        if not b: continue
        out = ngu.run_network(s, b["input"])
        if not np.array_equal(out, b["output"]): nf += 1
    tp = s.end_profiling()
    mem, par = ngu.score_network(san, tp)
    import os
    try: os.remove(tp)
    except: pass
    cost = (mem + par) if (mem is not None and par is not None) else None
    pts = max(1.0, 25.0 - math.log(max(1.0, cost))) if cost is not None else None
    return nf, cost, pts

print("verify_and_score ready — call it on YOUR OWN task001..task400 graph once you build one.")
print("(this notebook deliberately does not build one for you)")

## 7. How the cost/scoring actually works

**Per task:** your ONNX graph must produce the exact correct `output` for *every* train + test +
arc-gen example, or that task scores **0** — there's no partial credit for "mostly right." Once a
task is 100% correct, its score is a function of how cheap the graph is:

```
cost   = memory + params
points = max(1, 25 - ln(cost))          # per task, so max 25/task
```

**Total competition score** = sum of `points` over all 400 tasks (max 10,000). This is why the
whole "golfing" exercise exists: correctness alone caps you at 25 points/task, and squeezing
`cost` down is the only way to move within that cap — and because `cost` is inside a natural
log, cheap gets cheaper for less and less marginal benefit (cost 10->5 is worth more than
10,000->9,995).

Two terms make up `cost`, straight from `neurogolf_utils.py::calculate_memory` /
`calculate_params`, and they are **not** in the same units — that's the detail people miss:

- **`memory`** (bytes) — for every **intermediate** tensor (any node output whose name isn't
  `"input"` or `"output"`), `num_elements * dtype_itemsize`, using the max size seen across every
  profiled run. `input`/`output` themselves are *exempt* — only what happens **between** them
  costs anything.
- **`params`** (raw element count, not bytes!) — total element count across every
  `initializer` (and any `Constant` node's value). A `[10]` int64 table is 10 params whether it's
  int64 or int8; dtype doesn't matter here, only element count.
- `cost = memory_bytes + params_elements` — literally added together despite the unit mismatch.
  That's exactly why a `[10]`-element `Gather` index table is so cheap relative to a memory-heavy
  intermediate tensor: 10 raw elements versus tensors measured in bytes-times-thousands.

### Worked example: predict the cost by hand, then check it

Take the smallest graph with an actual intermediate tensor: `Mul(input, two) -> a`, then
`Identity(a) -> output`, on the real `(1,10,30,30)` NeuroGolf shape.

```mermaid
flowchart LR
    input(["input\n(1,10,30,30) float32\nEXEMPT"]) --> Mul
    two[("two\nscalar float32\ninitializer")] --> Mul
    Mul --> a["a\n(1,10,30,30) float32\nCOUNTS toward memory"]
    a --> Identity
    Identity --> output(["output\nEXEMPT"])
```

By hand, before running anything:
- `a` has shape `(1,10,30,30)` float32 -> `1*10*30*30 = 9000` elements * 4 bytes = **36000 bytes**
  of memory. (`input` and `output` don't count, even though `output` is literally `Identity`'s
  result — the exemption is by *name*, not by position in the graph.)
- `two` is a 0-d (scalar) initializer -> **1** param.
- Predicted `cost = 36000 + 1 = 36001`, `points = max(1, 25 - ln(36001)) ≈ 25 - 10.491 ≈ 14.509`.

Now build it and let the real scorer confirm the hand-computed numbers exactly:


In [ ]:
def score_model(model, sample_input=None):
    # memory + params for any single-input/single-output static-shape model. No task needed.
    san = ngu.sanitize_model(copy.deepcopy(model))
    onnx.checker.check_model(san, full_check=True)
    o = onnxruntime.SessionOptions(); o.enable_profiling = True; o.log_severity_level = 3
    o.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
    o.profile_file_prefix = "tutorial_score"
    s = onnxruntime.InferenceSession(san.SerializeToString(), o)
    if sample_input is None:
        in_info = san.graph.input[0]
        shape = [d.dim_value for d in in_info.type.tensor_type.shape.dim]
        sample_input = np.zeros(shape, np.float32)
    s.run(None, {san.graph.input[0].name: sample_input})
    tp = s.end_profiling()
    mem, par = ngu.score_network(san, tp)
    import os
    try: os.remove(tp)
    except: pass
    cost = (mem + par) if (mem is not None and par is not None) else None
    pts = max(1.0, 25.0 - math.log(max(1.0, cost))) if cost is not None else None
    return mem, par, cost, pts

shape = [1, 10, 30, 30]
X = helper.make_tensor_value_info('input', TensorProto.FLOAT, shape)
Y = helper.make_tensor_value_info('output', TensorProto.FLOAT, shape)

m_handcheck = graph_from(
    [helper.make_node('Mul', ['input', 'two'], ['a']),
     helper.make_node('Identity', ['a'], ['output'])],
    [numpy_helper.from_array(np.array(2.0, np.float32), 'two')], [X], [Y])

mem, par, cost, pts = score_model(m_handcheck)
print(f"memory={mem}  params={par}  cost={cost}  points={pts:.6f}")
assert mem == 36000 and par == 1 and cost == 36001
assert abs(pts - (25.0 - math.log(36001))) < 1e-9
print("matches the hand computation exactly")

Now the payoff: two graphs that both correctly reverse the 10 color channels of a
`(1,10,30,30)` tensor, with **no intermediate tensor at all** (each is a single node straight
from `input` to `output`, so `memory = 0` for both — the entire cost difference comes from
`params`, i.e. how many raw elements the chosen node's initializer needs):

- **`Gather`** with a `[10]` int64 permutation index -> `params = 10` -> `cost = 10` ->
  `points = 25 - ln(10) ≈ 22.697`.
- **`Conv`** with a `[10,10,1,1]` permutation *matrix* -> `params = 100` -> `cost = 100` ->
  `points = 25 - ln(100) ≈ 20.395`.

Same correctness, same zero memory cost, but the `Gather` version's initializer has 10x fewer
elements — predict those two point values yourself, then run the cell and check.


In [ ]:
# option A: Gather with a reversed channel index -- cost ~= 10 params
perm = np.arange(9, -1, -1, dtype=np.int64)
m_gather = graph_from([helper.make_node('Gather', ['input', 'perm'], ['output'], axis=1)],
                       [numpy_helper.from_array(perm, 'perm')], [X], [Y])

# option B: a 10x10x1x1 Conv permutation matrix -- cost ~= 100 params, same result
W = np.zeros((10, 10, 1, 1), np.float32)
for c in range(10): W[9 - c, c, 0, 0] = 1.0
m_conv = graph_from([helper.make_node('Conv', ['input', 'W'], ['output'], kernel_shape=[1, 1])],
                     [numpy_helper.from_array(W, 'W')], [X], [Y])

for name, mm in [('Gather permutation', m_gather), ('Conv permutation matrix', m_conv)]:
    mem, par, cost, pts = score_model(mm)
    print(f"{name:28s} memory={mem:>6}  params={par:>4}  cost={cost:>6}  points={pts:.3f}")
    assert mem == 0   # no intermediate tensor in either graph -- the whole gap is in `params`

Same correctness, and the `Gather` version scores meaningfully higher — that's the whole
game of "golfing" a graph: find the *cheapest node combination* that's still exactly correct on
every train/test/arc-gen example. `build_recolor` in `from_scratch.py` makes exactly this
`Gather`-vs-`Conv` decision for real, based on whether a task's color map happens to be a pure
bijection (`Gather` works) or not (needs the `Conv` channel-sum matrix instead).


## 8. Exercises — your turn

Each exercise gives you a numpy expression and an *empty* graph-building cell with `check()`
already wired up. Fill in `nodes`/`inits` so the assertion passes. No solutions are provided —
work through the phrasebook above if you get stuck on which op to reach for.


**Exercise 1** — build the ONNX graph for `np.rot90(x, k=2)` (180° rotation) on a
`(4,4)` float array. Hint: 180° rotation reverses both axes — think about what you did for
flip, applied twice.

In [ ]:
x = np.arange(16, dtype=np.float32).reshape(4, 4)
expected = np.rot90(x, k=2)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [4, 4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [4, 4])
inits = []   # TODO: fill in
nodes = []   # TODO: fill in

# m = graph_from(nodes, inits, [X], [Y])
# check('rot90(k=2)', expected, run(m, {'X': x}))

**Exercise 2** — build the graph for: *replace every occurrence of the value `3` with
`0`, and every occurrence of `0` with `3`* (a color swap), on a `(5,5)` float array with values
in `0..9`. Hint: this is a 1:1 recolor table — which single phrasebook op is built exactly for
that?

In [ ]:
rng = np.random.RandomState(2)
x = rng.randint(0, 10, size=(5, 5)).astype(np.float32)
swap = {3: 0, 0: 3}
expected = np.vectorize(lambda v: swap.get(int(v), int(v))).__call__(x).astype(np.float32)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [5, 5])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [5, 5])
inits = []   # TODO: fill in
nodes = []   # TODO: fill in

# m = graph_from(nodes, inits, [X], [Y])
# check('swap colors 0<->3', expected, run(m, {'X': x}))

**Exercise 3** — build the graph for: *count how many cells in each row are nonzero*
(`(x != 0).sum(axis=1)`), on a `(4,6)` float array. Hint: comparison -> Cast -> reduction,
three phrasebook pieces chained.

In [ ]:
rng = np.random.RandomState(3)
x = (rng.rand(4, 6) > 0.5).astype(np.float32) * rng.randint(1, 10, size=(4, 6))
expected = (x != 0).sum(axis=1).astype(np.float32)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [4, 6])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [4])
inits = []   # TODO: fill in
nodes = []   # TODO: fill in

# m = graph_from(nodes, inits, [X], [Y])
# check('count nonzero per row', expected, run(m, {'X': x}))

**Exercise 4 (harder, combines multiple sections)** — build the graph for: *crop the
top-left `3x3` of a `(6,6)` grid, upscale it 2x in both directions (back to `6x6`), then pad a
1-pixel border of zeros is NOT needed since it's already `6x6` — instead, recolor every cell
equal to `0` to `5`.* Chain: `Slice` -> upscale (`Reshape`/`Expand`/`Reshape`) -> recolor
(`Equal`+`Where`, or a `Gather` table).

In [ ]:
rng = np.random.RandomState(4)
x = rng.randint(0, 10, size=(6, 6)).astype(np.float32)
cropped = x[:3, :3]
upscaled = np.repeat(np.repeat(cropped, 2, axis=0), 2, axis=1)
expected = np.where(upscaled == 0, 5.0, upscaled).astype(np.float32)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [6, 6])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [6, 6])
inits = []   # TODO: fill in
nodes = []   # TODO: fill in

# m = graph_from(nodes, inits, [X], [Y])
# check('crop -> upscale 2x -> recolor 0->5', expected, run(m, {'X': x}))

### Where to go from here

- Work through Exercises 1-4 above (uncomment the `m = ...` / `check(...)` lines once you fill
  in `nodes`/`inits`).
- Then open `from_scratch.ipynb` / `from_scratch.py` — it has real, scored graphs for a handful
  of tasks (001, 003, 006, 016, 385, plus generic `build_recolor`/`build_geom`/`build_upscale`/
  `build_crop` builders) that you can study for pattern ideas once you're stuck, but the actual
  per-task algorithm-finding is on you from here.
- `draw_onnx()` / `render_mermaid(onnx_to_mermaid(...))` (Section 4) and `verify_and_score()`
  (Section 6) are the functions you'll call constantly while iterating on a real task graph.
